# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
from dataclasses import dataclass, field
from lib.vector_db import VectorStoreManager, CorpusLoaderService
import chromadb
from typing import List, Dict, Optional, TypedDict, Union
from openai import OpenAI
import numpy as np
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from tavily import TavilyClient
from datetime import datetime
from pydantic import BaseModel
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run
from lib.tooling import Tool, ToolCall, tool
import json


In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
class GameInfo(BaseModel):
    """
    Represents a single piece of memory information stored in the long-term memory system.
    
    This class encapsulates information about video games stored in the crhomadb that can be
    retrieved later to provide personalized responses in conversational AI applications.
    
    Attributes:
        Platform (str): like Game Boy, Playstation 5, Xbox 360...)
        Name (str): Name of the Game
        YearOfRelease (str):  Year when that game was released for that platform
        Description (str): Additional details about the game
    """
    Platform: str
    Name: str
    YearOfRelease: str
    Description: str 

@tool
def retrieve_game(question: str) -> List[GameInfo]:
    """
    Performs a semantic search against the 'udaplay' vector database to find 
    information about video games based on a natural language question.

    This function manually generates an embedding for the input question using 
    OpenAI's 'text-embedding-3-large' model to ensure compatibility with the 
    stored vector dimensions (3072) and retrieves the most relevant game record.

    Args:
        question (str): A natural language query or question about the game 
            industry, specific video games, or game platforms.

    Returns:
        List[GameInfo]: A list containing a GameInfo object if a match is found, 
            otherwise an empty list. Each object includes the game's Name, 
            Platform, YearOfRelease, and Description.
    """
    chroma_client = chromadb.PersistentClient(path="/workspace/Code/project/chromadb")
    collection = chroma_client.get_collection(name="udaplay")

    client = OpenAI(
        api_key=OPENAI_API_KEY,
        base_url="https://openai.vocareum.com/v1"
    )
    
    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    )
    question_embedding = response.data[0].embedding
    
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=5 
    )

    if not results['metadatas'] or len(results['metadatas'][0]) == 0:
        return []

    meta = results['metadatas'][0][0]
    
    return [GameInfo(
        Platform=meta.get("Platform", "N/A"),
        Name=meta.get("Name", "N/A"),
        YearOfRelease=str(meta.get("YearOfRelease", "N/A")),
        Description=meta.get("Description", "N/A")
    )]


#### Evaluate Retrieval Tool

In [5]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> EvaluationReport:
    """
    Based on the user's question and the retrieved documents, it analyzes 
    the usability of the documents to respond to that question.
    """
    client = OpenAI(api_key=OPENAI_API_KEY, base_url="https://openai.vocareum.com/v1")
    
    prompt = f"""
    Your task is to evaluate if the retrieved documents are enough to respond to the user query.
    
    User Query: {question}
    Retrieved Documents: {retrieved_docs}
    
    Analyze if the content actually answers the user's intent. 
    Give a detailed explanation of why it is or isn't useful.
    """

    # Using structured outputs/parsing to get the EvaluationReport
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a retrieval quality evaluator."},
            {"role": "user", "content": prompt},
        ],
        response_format=EvaluationReport,
    )
    
    return completion.choices[0].message.parsed.model_dump()

#### Game Web Search Tool

In [6]:
@tool
def game_web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)

    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    
    return formatted_results

### Agent

In [7]:
class AgentState(TypedDict):
    user_query: str  
    instructions: str  
    messages: List[dict] 
    current_tool_calls: Optional[List[ToolCall]]  


class Agent:
    def __init__(self, 
                 model_name: str,
                 instructions: str, 
                 tools: List[Tool] = None,
                 temperature: float = 0.7):
        """
        Initialize an Agent instance
        
        Args:
            model_name: Name/identifier of the LLM model to use
            instructions: System instructions for the agent
            tools: Optional list of tools available to the agent
            temperature: Temperature parameter for LLM (default: 0.7)
        """
        self.instructions = instructions
        self.tools = tools if tools else []
        self.model_name = model_name
        self.temperature = temperature
                
        # Initialize state machine
        self.workflow = self._create_state_machine()

    def _prepare_messages_step(self, state: AgentState) -> AgentState:
        """Step logic: Prepare messages for LLM consumption"""

        messages = [
            SystemMessage(content=state["instructions"]),
            UserMessage(content=state["user_query"])
        ]
        
        return {
            "messages": messages
        }

    def _llm_step(self, state: AgentState) -> AgentState:
        """Step logic: Process the current state through the LLM"""

        # Initialize LLM
        llm = LLM(
            model=self.model_name,
            temperature=self.temperature,
            tools=self.tools
        )

        response = llm.invoke(state["messages"])
        tool_calls = response.tool_calls if response.tool_calls else None

        # Create AI message with content and tool calls
        ai_message = AIMessage(content=response.content, tool_calls=tool_calls)
        
        return {
            "messages": state["messages"] + [ai_message],
            "current_tool_calls": tool_calls
        }

    def _tool_step(self, state: AgentState) -> AgentState:
        tool_calls = state["current_tool_calls"] or []
        tool_messages = []
        
        for call in tool_calls:
            function_name = call.function.name
            function_args = json.loads(call.function.arguments)
            
            tool = next((t for t in self.tools if t.name == function_name), None)
            if tool:
                result = tool(**function_args)
                
                # --- Enforce conversion to dict for JSON safety ---
                if isinstance(result, list):
                    # This turns your List[GameInfo] into List[dict]
                    safe_content = [item.model_dump() if isinstance(item, BaseModel) else item for item in result]
                elif isinstance(result, BaseModel):
                    safe_content = result.model_dump()
                else:
                    safe_content = result

                tool_messages.append(
                    ToolMessage(
                        content=json.dumps(safe_content), # This is now pure JSON
                        tool_call_id=call.id, 
                        name=function_name, 
                    )
                )
        
        # We return a clean list of messages to the StateMachine
        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None
        }

    def _create_state_machine(self) -> StateMachine[AgentState]:
        """Create the internal state machine for the agent"""
        machine = StateMachine[AgentState](AgentState)
        
        # Create steps
        entry = EntryPoint[AgentState]()
        message_prep = Step[AgentState]("message_prep", self._prepare_messages_step)
        llm_processor = Step[AgentState]("llm_processor", self._llm_step)
        tool_executor = Step[AgentState]("tool_executor", self._tool_step)
        termination = Termination[AgentState]()
        
        machine.add_steps([entry, message_prep, llm_processor, tool_executor, termination])
        
        # Add transitions
        machine.connect(entry, message_prep)
        machine.connect(message_prep, llm_processor)
        
        # Transition based on whether there are tool calls
        def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
            """Transition logic: Check if there are tool calls"""
            if state.get("current_tool_calls"):
                return tool_executor
            return termination
        
        machine.connect(llm_processor, [tool_executor, termination], check_tool_calls)
        machine.connect(tool_executor, llm_processor)  # Go back to llm after tool execution
        
        return machine

    def invoke(self, query: str) -> Run:
        """
        Run the agent on a query
        
        Args:
            query: The user's query to process
            
        Returns:
            The final run object after processing
        """

        initial_state: AgentState = {
            "user_query": query,
            "instructions": self.instructions,
            "messages": [],
        }

        run_object = self.workflow.run(initial_state)

        return run_object


tools = [retrieve_game, evaluate_retrieval, game_web_search]

In [8]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?


research_agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=(
        "You are an AI Research Agent specialized in the video game industry. "
        "Your goal is to provide accurate information about games, platforms, and industry trends. "
        
        "### Operational Workflow:\n"
        "1. **Internal Search**: Always start by searching the internal database using the retrieval tool.\n"
        "2. **Evaluation**: Once you get a result, you MUST call 'evaluate_retrieval' to check if the data is useful.\n"
        "3. **Web Search**: If 'evaluate_retrieval' reports the info is missing or low quality, call 'game_web_search'.\n"
        
        "### Execution Pattern:\n"
        "- Use **Thought** to plan your research steps based on the user's question.\n"
        "- Use **Action** to call the specific tool required.\n"
        "- Use **Observation** to analyze the tool output before moving to the next step.\n"
        
        "### Key Constraints:\n"
        "- Never guess. If internal data fails evaluation, use the web.\n"
        "- Maintain a professional, research-oriented tone.\n"
        "- Return final answers in a structured format as requested by the user.\n"
        "- If information is found that is valuable for the future, explicitly state it in your final response.\n"
        
        f"Available Tools: {tools}"
    )
)

In [43]:
run_object = research_agent.invoke(
    query="When Pokémon Gold and Silver was released?"
)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [44]:
run_object.get_final_state()["messages"]

[SystemMessage(role='system', content="You are an AI Research Agent specialized in the video game industry. Your goal is to provide accurate information about games, platforms, and industry trends. ### Operational Workflow:\n1. **Internal Search**: Always start by searching the internal database using the retrieval tool.\n2. **Evaluation**: Once you get a result, you MUST call 'evaluate_retrieval' to check if the data is useful.\n3. **Web Search**: If 'evaluate_retrieval' reports the info is missing or low quality, call 'game_web_search'.\n### Execution Pattern:\n- Use **Thought** to plan your research steps based on the user's question.\n- Use **Action** to call the specific tool required.\n- Use **Observation** to analyze the tool output before moving to the next step.\n### Key Constraints:\n- Never guess. If internal data fails evaluation, use the web.\n- Maintain a professional, research-oriented tone.\n- Return final answers in a structured format as requested by the user.\n- If i

In [45]:
run_object = research_agent.invoke(
    query="Which one was the first 3D platformer Mario game?"
)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [46]:
run_object.get_final_state()["messages"]

[SystemMessage(role='system', content="You are an AI Research Agent specialized in the video game industry. Your goal is to provide accurate information about games, platforms, and industry trends. ### Operational Workflow:\n1. **Internal Search**: Always start by searching the internal database using the retrieval tool.\n2. **Evaluation**: Once you get a result, you MUST call 'evaluate_retrieval' to check if the data is useful.\n3. **Web Search**: If 'evaluate_retrieval' reports the info is missing or low quality, call 'game_web_search'.\n### Execution Pattern:\n- Use **Thought** to plan your research steps based on the user's question.\n- Use **Action** to call the specific tool required.\n- Use **Observation** to analyze the tool output before moving to the next step.\n### Key Constraints:\n- Never guess. If internal data fails evaluation, use the web.\n- Maintain a professional, research-oriented tone.\n- Return final answers in a structured format as requested by the user.\n- If i

In [9]:
run_object = research_agent.invoke(
    query="Was Mortal Kombat X realeased for Playstation 5?"
)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [10]:
run_object.get_final_state()["messages"]

[SystemMessage(role='system', content="You are an AI Research Agent specialized in the video game industry. Your goal is to provide accurate information about games, platforms, and industry trends. ### Operational Workflow:\n1. **Internal Search**: Always start by searching the internal database using the retrieval tool.\n2. **Evaluation**: Once you get a result, you MUST call 'evaluate_retrieval' to check if the data is useful.\n3. **Web Search**: If 'evaluate_retrieval' reports the info is missing or low quality, call 'game_web_search'.\n### Execution Pattern:\n- Use **Thought** to plan your research steps based on the user's question.\n- Use **Action** to call the specific tool required.\n- Use **Observation** to analyze the tool output before moving to the next step.\n### Key Constraints:\n- Never guess. If internal data fails evaluation, use the web.\n- Maintain a professional, research-oriented tone.\n- Return final answers in a structured format as requested by the user.\n- If i

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes